In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src" / "custseg").exists():   # project root = folder that holds the package
    if ROOT == ROOT.parent:
        raise RuntimeError("run the notebook from inside the project folder")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
from custseg import config, db
img = lambda p, w=900: display(Image(filename=str(ROOT / p), width=w))

# 01 · Data Profiling
Profile of the raw Olist files as loaded into schema `raw` (all TEXT).

In [2]:
man = json.loads((ROOT / '02_data/dataset_manifest.json').read_text())
pd.DataFrame({k: {'rows': v['rows'], 'columns': len(v['columns']), 'MB': round(v['bytes']/1e6, 1)} for k, v in man['files'].items()}).T

,rows,columns,MB
customers,99441.0,5.0,9.0
geolocation,1000163.0,5.0,61.3
order_items,112650.0,7.0,15.4
order_payments,103886.0,5.0,5.8
order_reviews,99224.0,7.0,14.5
orders,99441.0,8.0,17.7
products,32951.0,9.0,2.4
sellers,3095.0,4.0,0.2
category_translation,71.0,2.0,0.0


In [3]:
prof = pd.read_csv(ROOT / '04_data_profiling_eda/reports/data_profiling.csv')
prof[['table','column','inferred_type','null_pct','distinct','min','p50','max']]

,table,column,inferred_type,null_pct,distinct,min,p50,max
0,customers,customer_id,text,0.000000,99441,NaN,NaN,NaN
1,customers,customer_unique_id,text,0.000000,96096,NaN,NaN,NaN
2,customers,customer_zip_code_prefix,numeric,0.000000,14994,1003.0,24416.000000,99990.0
3,customers,customer_city,text,0.000000,4119,NaN,NaN,NaN
4,customers,customer_state,text,0.000000,27,NaN,NaN,NaN
5,geolocation,geolocation_zip_code_prefix,numeric,0.000000,19015,1001.0,26530.000000,99990.0
6,geolocation,geolocation_lat,numeric,0.000000,717372,-36.6053744107061,-22.919377,45.06593318269697
7,geolocation,geolocation_lng,numeric,0.000000,717615,-101.46676644931476,-46.637879,121.10539381057764
8,geolocation,geolocation_city,text,0.000000,8011,NaN,NaN,NaN
9,geolocation,geolocation_state,text,0.000000,27,NaN,NaN,NaN


In [4]:
db.query("SELECT count(*) orders, count(DISTINCT customer_id) customer_ids FROM raw.orders")

,orders,customer_ids
0,99441,99441


In [5]:
db.query("SELECT count(DISTINCT customer_unique_id) persons FROM raw.customers")

,persons
0,96096


**Takeaway:** `customer_id` is per order; the person is `customer_unique_id`. See `reports/data_profiling_report.md` for the full profile.